# 10.01 · Why evals & error analysis

**Chapter:** Evaluation · **Milestone:** M5

## Goals
- [ ] Understand why "vibe checks" don't scale and what an eval actually is
- [ ] Do **error analysis**: run the system, read traces, label failures, build a failure taxonomy, prioritise
- [ ] Separate failure *locations* (retrieval vs generation) so you fix the right component

## Resources
- [Hamel Husain – Your AI product needs evals](https://hamel.dev/blog/posts/evals/) · [A field guide to rapidly improving AI products](https://hamel.dev/blog/posts/field-guide/) — free
- [Shreya Shankar et al. – Who Validates the Validators?](https://arxiv.org/abs/2404.12272) — free
- [Hamel & Shreya – AI Evals course (Maven)](https://maven.com/parlance-labs/evals) — **paid**, the most complete course on this topic

**An eval** = a dataset of inputs (+ expectations) × a way to run your system × a way to score outputs × a number you track over time.
Without one, every prompt change is a guess and every model upgrade is a gamble. **Error analysis comes first**: you can't write good
metrics until you know how your system actually fails.

In [ ]:
import json
import re
from collections import Counter

import pandas as pd

from aieng import DATA, rag

pd.set_option("display.max_colwidth", 120)

## 1. Run the system and log everything
Each record is a **trace**: input, intermediate steps (retrieved ids), output. Save them — they're your raw material.

In [ ]:
evalset = rag.eval_set()[:40]
traces = []
for row in evalset:
    text, hits = rag.answer(row["question"], k=5, max_tokens=300)
    traces.append({"question": row["question"], "gold_answer": row["answer"], "retrieved": [pid for pid, _, _ in hits],
                   "gold_passages": sorted(row["gold"])[:10], "answer": text})
(DATA / "10-01-traces.jsonl").write_text("\n".join(json.dumps(t) for t in traces))
df = pd.DataFrame(traces)
df[["question", "gold_answer", "answer"]].head(8)

## 2. Locate failures automatically where you can
Cheap structural signals tell you *where* things broke before you read anything:
- **retrieval miss**: no gold passage in the retrieved set → fix retrieval (ch. 07–08), not the prompt
- **abstained**: model said "I don't know" — correct or not?
- **answer mismatch**: gold passage retrieved, but the answer doesn't contain the gold string → generation problem (or a paraphrase!)

In [ ]:
def auto_label(t):
    retrieved_gold = bool(set(t["retrieved"]) & set(t["gold_passages"])) or bool(set(t["retrieved"]) & next(r["gold"] for r in evalset if r["question"] == t["question"]))
    contains = t["gold_answer"].lower().rstrip(".") in t["answer"].lower()
    abstained = "don't know" in t["answer"].lower()
    if contains:
        return "ok (string match)"
    if not retrieved_gold:
        return "retrieval miss"
    if abstained:
        return "abstained despite evidence"
    return "answer mismatch (read it!)"


df["auto_label"] = [auto_label(t) for t in traces]
print(df["auto_label"].value_counts())

## 3. Read the failures — *open coding*
This is the step everyone skips and the one that matters most. Read each failing trace and write a short note on what went wrong, in your
own words. Don't use a predefined list yet. Below we print the "answer mismatch" cases; many will turn out to be **correct paraphrases**
(the metric is wrong, not the system) — that itself is a finding.

In [ ]:
for _, r in df[df["auto_label"].str.startswith("answer mismatch")].head(8).iterrows():
    print(f"Q: {r.question}\n  gold: {r.gold_answer!r}\n  got : {r.answer[:220]!r}\n")

## 4. Annotate, then group into a taxonomy (*axial coding*)
Write your notes into a CSV (or a spreadsheet / annotation tool: Argilla, Label Studio, Langfuse/Phoenix annotation queues), then cluster
notes into categories and count. Below: a starter file with an empty `note` column for you to fill, and an example taxonomy.

In [ ]:
annot_path = DATA / "10-01-annotations.csv"
if not annot_path.exists():
    df.assign(note="", category="")[["question", "gold_answer", "answer", "auto_label", "note", "category"]].to_csv(annot_path, index=False)
print("annotate here:", annot_path)

EXAMPLE_TAXONOMY = {
    "metric false negative": "answer is correct but phrased differently than gold",
    "retrieval miss": "relevant passage not retrieved",
    "partial answer": "answers part of a multi-part question",
    "unsupported claim": "adds facts not in the sources (hallucination)",
    "over-abstention": "says I don't know though the sources contain the answer",
    "bad gold label": "the dataset's answer is wrong/ambiguous",
}
for k, v in EXAMPLE_TAXONOMY.items():
    print(f"- {k}: {v}")

## 5. Prioritise
Count categories, weigh by user impact, fix the biggest first. Then turn each important category into an **automated check** (10.03):
a code assertion where possible, an LLM judge where necessary. The checks become your regression suite (10.05).

In [ ]:
annotated = pd.read_csv(annot_path).fillna("")
if annotated["category"].str.len().sum():
    print(annotated["category"].value_counts())
else:
    print("No categories yet — fill the CSV, rerun this cell. Meanwhile, auto-labels:", dict(Counter(df["auto_label"])))

## The eval mindset
- **Look at the data** before choosing metrics. Generic metrics ("helpfulness 4.2/5") hide what's broken.
- **Binary pass/fail judgements** with a written critique beat 1–5 scales: easier to label consistently, easier to act on.
- **Small and real** beats big and synthetic: 50 carefully-labelled real cases are worth more than 5,000 generated ones.
- **Evals are never done**: production traffic creates new failure modes (11.05).

## Exercises
1. Fill `note` and `category` for all 40 traces. Which category is biggest? Was it what you expected?
2. Rerun with a hosted model (`AIENG_PROVIDER=gemini`). Which categories shrink, which don't? (Retrieval misses won't — a bigger model can't fix them.)
3. Write one sentence per category describing the automated check you'd build for it.

In [ ]:
# your code here